# Epic Orchestrator — Analyse

Tier-1 eval (#390). Hinweis: der historische `EpicOrchestrator` wurde
refactored. Heute testbar: `TaskComplexityClassifier` isoliert + Plan-Runden
via `plan_and_react`-Strategie.

## 1. Setup

In [ ]:
import os, sys, importlib
from pathlib import Path

PYTF_ROOT = Path.cwd()
if PYTF_ROOT.name == 'notebooks':
    PYTF_ROOT = PYTF_ROOT.parent
os.chdir(PYTF_ROOT)
for p in [str(PYTF_ROOT / 'src'), str(PYTF_ROOT / 'notebooks')]:
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    from dotenv import load_dotenv
    load_dotenv(PYTF_ROOT / '.env')
except ImportError:
    pass

try:
    import nest_asyncio; nest_asyncio.apply()
except ImportError:
    pass

import logging, structlog
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s | %(message)s')
structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))

import analysis_lib
importlib.reload(analysis_lib)
import analysis_lib as alib
print('analysis_lib OK -', len(alib.list_evaluators()), 'evaluators')

In [ ]:
from taskforce.application.factory import AgentFactory
from taskforce.application.executor import AgentExecutor

factory = AgentFactory()
executor = AgentExecutor(factory=factory)
alib.disable_post_mission_learning(executor)
print('factory + executor ready')

## 2. TaskComplexityClassifier direkt aufsetzen

Der EpicOrchestrator wurde refactored — heute läuft die Eskalation über
`planning_strategy='plan_and_execute'` bzw. `plan_and_react`. Dieses Notebook
testet (a) den Classifier isoliert, (b) das Planning-Round-Verhalten an einem
Agent mit plan_and_react.

In [ ]:
WORKDIR = Path('.taskforce_epic_analysis')
WORKDIR.mkdir(exist_ok=True)

from taskforce_coding_agent.task_complexity_classifier import TaskComplexityClassifier
from taskforce.infrastructure.llm.litellm_service import LiteLLMService

# Classifier needs an LLM provider. Use the framework's LiteLLMService
# with the 'fast' model alias from llm_config.yaml.
llm_provider = LiteLLMService()
classifier = TaskComplexityClassifier(llm_provider, classification_model='fast')

TRIVIAL = ['Was ist 2+2?', 'Welche Uhrzeit haben wir gerade?']
COMPLEX = [
    'Baue ein REST-API für User-Management mit Auth, Tests und CI-Pipeline.',
    'Refactor unseren Wiki-Tool damit es Caching + Versionierung kann.',
]
for t in TRIVIAL + COMPLEX:
    try:
        result = await classifier.classify(t)
        out = f'{result.level}  conf={result.confidence:.2f}  reason={result.reason[:40]}'
    except Exception as exc:
        out = f'ERROR: {type(exc).__name__}: {exc}'[:80]
    print(f'  {t[:60]:60s} -> {out}')

## 3. Plan-and-React Agent + Rounds-Beobachtung

Agent mit plan_and_react: jeder planner-Tool-Call = neue Plan-Runde.

In [ ]:
EPIC_TOOLS = ['python', 'file_read', 'web_search']
EPIC_PROMPT = (
    'Du löst Aufgaben mit einem Plan-and-React-Loop. Erstelle für nicht-triviale '
    'Aufgaben einen Plan via planner-Tool, dann arbeite ihn Step-by-step ab.'
)

async def build_epic_agent():
    a = await factory.create_agent(
        system_prompt=EPIC_PROMPT, tools=EPIC_TOOLS,
        persistence={'type': 'file', 'work_dir': str(WORKDIR)},
        work_dir=str(WORKDIR),
        planning_strategy='plan_and_react',
        planning_strategy_params={'max_plan_steps': 6},
        max_steps=12,
    )
    alib.patch_anti_compression(a)
    return a, len(EPIC_PROMPT)

BUILD_AGENT = build_epic_agent
_smoke, _chars = await BUILD_AGENT()
AVAILABLE_TOOLS = set(_smoke.tools.keys())
print(f'tools={sorted(AVAILABLE_TOOLS)}')
await _smoke.close()

## 3. Szenarien laden

Aus `scenarios/epic.yaml`.

In [ ]:
all_scenarios = alib.load_scenarios('notebooks/scenarios/epic.yaml')
eligible = alib.filter_scenarios(all_scenarios, AVAILABLE_TOOLS)
print(f'Total: {len(all_scenarios)}, Eligible: {len(eligible)}')
for s in eligible:
    print(f'  - {s.id:35s} ({s.category}/{s.difficulty})')

## 4. Einzelnes Szenario (Detail-Lauf)

In [ ]:
TARGET_ID = 'epic-trivial-no-epic'  # change me
target = next((s for s in eligible if s.id == TARGET_ID), None) or eligible[0]
print(f'Target: {target.id}')
print(f'Mission: {target.mission}')
print(f'Hidden intent: {target.hidden_intent}')

In [ ]:
agent, sys_chars = await BUILD_AGENT()
rec = await alib.run(
    executor, agent, target.mission,
    project_root=WORKDIR, snapshot_subdirs=(),
    initial_system_prompt_chars=sys_chars, silent=False,
)
# Stash MCP tool names if any (no-op for non-MCP notebooks)
rec.extra['mcp_tool_names'] = list(_mcp_tool_names_of(agent)) if '_mcp_tool_names_of' in dir() else []
alib.print_summary(rec)

## 5. Reports

In [ ]:
alib.print_tool_stats(rec)
print()
alib.print_tool_results(rec, head=15)

In [ ]:
card = alib.score_rule_based(rec, target)
print(f'=== Rule-based ({"PASS" if card.passed else "FAIL"}) ===')
alib.print_feature_checks(card)
if card.details:
    print('Details:')
    for d in card.details:
        print(f'  - {d}')

In [ ]:
await agent.close()
_ = alib.plot_tool_frequencies(rec, title=f'Tool-Aufrufe: {target.id}')

## 6. Batch-Lauf

In [ ]:
results = await alib.run_scenarios(
    executor, BUILD_AGENT, eligible,
    project_root=WORKDIR, snapshot_subdirs=(),
    reset_dirs_before_each=(),
    repeats=1, progress=True,
)  
print()
alib.print_scenario_summary(results)

In [ ]:
_ = alib.plot_scenario_matrix(results, metric='passed', title='Pass/Fail')
_ = alib.plot_scenario_matrix(results, metric='tool_calls', title='Tool calls')

## Ideen für weitere Experimente

Siehe TODOs in der Scenario-YAML.

In [ ]:
print('done')